In [ ]:
# IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report

# Loading labelled dataset
labeled_csv = "data-performance.csv"
df_labeled = pd.read_csv(labeled_csv)

# Encoding non-numerical data (2nd and lasr column - gender and class) by LabelEncoder
all_genders = pd.concat([df_labeled.iloc[:, 1]])  # Grouping of columns for consistent encoding
encoder = LabelEncoder()
encoder.fit(all_genders)
df_labeled.iloc[:, 1] = encoder.transform(df_labeled.iloc[:, 1].values)
y = encoder.fit_transform(df_labeled.iloc[:, -1].values) 
X = df_labeled.iloc[:, :-1].values

# Preprocessing of data using the StandardScaler to normalize the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Splitting data into training and testing sets with 0.2/0.8
test_size = 0.2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

# Training the model
model = SVC(kernel="rbf", C=0.7, class_weight="balanced", probability=True, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Creation of the Classification Report
classification_rep = classification_report(y_test, y_pred, target_names=encoder.classes_, output_dict=True)
df_classification_report = pd.DataFrame(classification_rep).transpose()
x = df_classification_report.to_csv()
print(x)

# Creation of the Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# Printing Cross-validation scores
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
cv_metrics = {
    "Mean Accuracy": [np.mean(cv_scores)],
    "Standard Deviation": [np.std(cv_scores)]
}
df_cv_scores = pd.DataFrame(cv_metrics)
y = df_cv_scores.to_csv()
print(y)